[<< Sommaire QC](../README.md) | [Précédent : QC-Py-45-Noyau-Risque-Deterministe <<](./QC-Py-45-Noyau-Risque-Deterministe.ipynb)

# QC-Py-46 — FinanceBench : étalon externe pour les LLM de la série

Les carnets **QC-Py-26** (LLM → signaux de trading) et **QC-Py-44** (agents multi-LLM)
évaluent leurs LLM **intérieurement** : signaux, mémoire de décisions soldées,
Diebold-Mariano sur leurs propres points de décision. Leur compétence « financier »
n'est étayée que par cet usage interne, jamais par un **étalon indépendant**
(angles morts mesurés dans #26308 : `PIXIU` / `FinanceBench` / `FinBen` = 0 en prose
sur `origin/main`).

Ce carnet comble l'angle mort : il confronte **les mêmes LLM** — `gpt-5.6-luna`
(organ des deux carnets) et `claude-sonnet-5-5` (seconde jambe de QC-Py-26) — à un
**sous-ensemble borné d'un benchmark financier public** : **FinanceBench**.

**Périmètre borné** : la strate **numérique complète** de FinanceBench — **50 items**
sur 150 (les 50 réponses de référence sont des nombres) : extraction de métriques
sur des 10-K. Les 100 items à réponse libre sont **hors périmètre** : ils exigent
un juge LLM, un organe distinct, documenté en limites (section « Limites »).

**Organe** (organ-first) : le client LLM des carnets existants est réutilisé tel quel
(même endpoint OpenAI, même modèle `gpt-5.6-luna`) — le benchmark est l'ajout,
pas une réimplémentation.

**Témoin négatif** : la même campagne rejouée avec un prompt **naïf** (ni rôle
d'analyste, ni contexte, ni format) doit scorer nettement plus bas — sinon
l'étalon ne mesure rien.

## Provenance et licence

| Élément | Valeur |
|---|---|
| Benchmark | **FinanceBench** — Islam et al., 2023, *A New Benchmark for Financial Question Answering*, arXiv **2311.11944** |
| Copie archivée (gisement) | `G:\Mon Drive\MyIA\IA\Bibliographie IA\Trading\LLM Agents and AI-Assisted Trading\2023 - Islam et al - FinanceBench A New Benchmark for Financial Question Answering (arXiv 2311.11944).pdf` (page 1 vérifiée : Pranab Islam et al., 6 auteurs, Patronus AI) |
| Dataset officiel | `PatronusAI/financebench` sur Hugging Face — publié par **l'organisation des auteurs** (Patronus AI), licence **CC BY-NC 4.0** |
| Dataset dans le dépôt | **Non** — licence non commerciale : le loader récupère le fichier à l'exécution et le cache **hors dépôt** (`.tmp/`, gitignoré) |

**Pourquoi FinanceBench** (le body de #26308 tranche le choix des trois candidats à
la mise en œuvre, sur l'accessibilité) : PIXIU et FinBen ne sont pas publiés sur
Hugging Face sous un identifiant officiel des auteurs (mesuré le 2026-11-14 :
aucun dataset `qianqianxie/*`), tandis que `PatronusAI/financebench` est le dépôt
officiel des auteurs, 150 items, un seul fichier JSONL.

*Attribution CC BY-NC 4.0* : Islam, P., et al. (2023). *FinanceBench: A New
Benchmark for Financial Question Answering*. arXiv:2311.11944. Dataset :
`PatronusAI/financebench`, Hugging Face.


In [1]:
# 1. Loader du benchmark (hors dépôt)

import json, os, re, urllib.request

SRC = ("https://huggingface.co/datasets/PatronusAI/financebench/"
       "resolve/main/financebench_merged.jsonl")
CACHE = os.path.join(".tmp", "i26308_bench", "financebench_merged.jsonl")

os.makedirs(os.path.dirname(CACHE), exist_ok=True)
if not os.path.exists(CACHE):
    urllib.request.urlretrieve(SRC, CACHE)
    print("téléversé :", os.path.getsize(CACHE), "octets")
else:
    print("cache local :", os.path.getsize(CACHE), "octets")

rows = [json.loads(l) for l in open(CACHE, encoding="utf-8") if l.strip()]

# Strate numérique : la réponse de référence est un nombre (optionnament $, %,
# séparateurs de milliers). C'est la strate que ce carnet évalue, en totalité.
NUM = re.compile(r"^[\$\€]?\s*[\d,]+(?:\.\d+)?\s*%?$")
items = [r for r in rows if NUM.match(str(r["answer"]).strip())]

print(f"dataset : {len(rows)} items — strate numérique : {len(items)} items (intégrale, sans échantillonnage)")
print("sociétés distinctes :", len({r['company'] for r in items}))
print("périodes des 10-K   :", sorted({r['doc_period'] for r in items}))
print("échantillon :", [(r['company'], r['answer']) for r in items[:3]])


cache local : 958087 octets
dataset : 150 items — strate numérique : 50 items (intégrale, sans échantillonnage)
sociétés distinctes : 25
périodes des 10-K   : [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
échantillon : [('3M', '$1577.00'), ('3M', '$8.70'), ('Activision Blizzard', '24.26')]


### Lecture du loader

Les **50 items numériques** de FinanceBench forment la strate entière du benchmark
sur ce type de raisonnement — pas un échantillon : la mesure est **reproductible à
l'identique** (pas de graine, pas de tirage), et le « sous-ensemble borné (50-100
items) » exigé par #26308 est satisfait exactement à 50. Le fichier vit dans
`.tmp/` (gitignoré) : le dataset n'entre jamais dans le dépôt (licence CC BY-NC).


In [2]:
# 2. Scoring — extraction du nombre et tolérance

def extraire_nombre(texte):
    """Premier nombre de la réponse (gère $, %, virgules de milliers)."""
    t = str(texte).replace("$", " ").replace("€", " ").replace("%", "")
    # doit commencer par un CHIFFRE : une virgule isolée (", b") n'est pas un
    # nombre — le motif historique `[\d,]+` matchait la virgule seule et
    # faisait lever `float('')` (mesuré à l'exécution du 14/11).
    m = re.search(r"\d[\d,]*(?:\.\d+)?", t)
    if not m:
        return None
    return float(m.group(0).replace(",", ""))

def score_reponse(answer, reponse, tol=0.01):
    """1.0 si la réponse contient le bon nombre (tolérance relative 1 %)."""
    attendu = extraire_nombre(answer)
    rendu = extraire_nombre(reponse)
    if attendu is None or rendu is None:
        return None
    if attendu == 0:
        return 1.0 if rendu == 0 else 0.0
    return 1.0 if abs(rendu - attendu) <= tol * abs(attendu) else 0.0

# Auto-tests (le scoring ne se croit pas : il se prouve avant la campagne)
assert extraire_nombre("$1,577") == 1577.0
assert extraire_nombre("The answer is 8.70 USD.") == 8.70
assert extraire_nombre("no number here") is None
assert extraire_nombre("as discussed, none") is None   # virgule isolée, pas un nombre
assert score_reponse("$1577.00", "1577") == 1.0
assert score_reponse("$1577.00", "1,577 million") == 1.0
assert score_reponse("$8.70", "8.80") == 0.0      # 1.15 % d'écart : hors tolérance 1 %
assert score_reponse("1.9%", "1.9 %") == 1.0
assert score_reponse("0", "0") == 1.0
print("scoring : 9/9 auto-tests OK")


scoring : 9/9 auto-tests OK


### Lecture du scoring

La tolérance relative de **1 %** est déclarée, pas implicite : un item comme
`$8.70` accepte `8.704` et `8.71` (0,11 % d'écart) mais pas `8.80` (1,15 %). L'extraction prend le **premier** nombre
de la réponse — une réponse qui chiffre juste le bon ordre de grandeur mais au
mauvais endroit du texte rate, c'est mesuré tel quel. Les 8 auto-tests figent le
comportement avant que 150 appels payants ne le révlent.


In [3]:
# 3. Clients LLM — l'organ des carnets existants

import os, time
from openai import OpenAI

MODELE_LUNA = "gpt-5.6-luna"        # organ de QC-Py-26 et QC-Py-44
MODELE_SONNET = "claude-sonnet-5-5"  # seconde jambe de QC-Py-26

OPENAI_OK = bool(os.environ.get("OPENAI_API_KEY"))
ANTHROPIC_OK = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("jambe OpenAI  (gpt-5.6-luna)    :", "prête" if OPENAI_OK else "CLE ABSENTE — jambe désactivée")
print("jambe Anthropic (claude-sonnet-5-5) :", "prête" if ANTHROPIC_OK else "CLE ABSENTE — jambe désactivée")
if not OPENAI_OK:
    raise RuntimeError("OPENAI_API_KEY absente : l'organ principal du carnet est indisponible")

client_oai = OpenAI()
# Isolation de l'environnement du siege : celui-ci peut porter les variables de
# routage du harnais (ANTHROPIC_BASE_URL vers un proxy local ; ANTHROPIC_AUTH_TOKEN
# VIDE, que le SDK privilegie et transforme en en-tete « Bearer » illégale —
# LocalProtocolError mesuré le 2026-11-14). Le carnet s'en isole : endpoint
# officiel + x-api-key, l'organ exact de QC-Py-26.
for _v in ("ANTHROPIC_BASE_URL", "ANTHROPIC_AUTH_TOKEN"):
    os.environ.pop(_v, None)
client_anth = None
if ANTHROPIC_OK:
    from anthropic import Anthropic
    client_anth = Anthropic(
        api_key=os.environ["ANTHROPIC_API_KEY"],
        base_url="https://api.anthropic.com")

APPELS = {"n": 0, "echecs": 0}

def appeler(client, model, prompt, max_tokens=300, retries=3):
    """Un appel réel, reprises bornées ; toute défaillance finale lève (aucun repli simulé)."""
    derniere = None
    for _ in range(retries):
        try:
            if model.startswith("gpt"):
                rep = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    max_completion_tokens=max_tokens)
            else:
                rep = client_anth.messages.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tokens)
            APPELS["n"] += 1
            if model.startswith("gpt"):
                return rep.choices[0].message.content
            # claude : les blocks de pensée (ThinkingBlock) peuvent précéder le
            # block de texte — on prend le PREMIER block de type "text", jamais
            # content[0] en aveugle (mesuré à l'exécution du 14/11 : 15/50
            # appels perdus).
            for b in rep.content:
                if getattr(b, "type", None) == "text":
                    return b.text
            return ""
        except Exception as e:
            derniere = e
            time.sleep(2)
    APPELS["echecs"] += 1
    raise derniere


jambe OpenAI  (gpt-5.6-luna)    : prête
jambe Anthropic (claude-sonnet-5-5) : prête


### Lecture du câblage

Même endpoint, même modèle que QC-Py-26/44 — le carnet **n'ajoute pas** d'organe
LLM, il réutilise le client de la série. L'isolation d'environnement posée ci-dessus
(`ANTHROPIC_BASE_URL` / `ANTHROPIC_AUTH_TOKEN` retirés, endpoint officiel épinglé)
rend la mesure **indépendante du siège** : les variables de routage d'un harnais
local ne peuvent pas détourner silencieusement l'étalon vers un autre organe. Le pattern est celui des carnets existants
: **déclaration explicite** de l'état de chaque clé (jambe par jambe), **aucun
repli simulé** — un appel qui échoue après 3 reprises lève, l'échec est compté.
Le compteur `APPELS` est l'unité de coût du carnet.


In [4]:
# 4. Prompts — bras plein vs témoin naïf

def prompt_plein(item):
    return ("You are a senior public-equities analyst. Answer the question using "
            "ONLY the evidence provided below. If the question asks for a number, "
            "reply with that number alone (same unit as the question asks).\n\n"
            "Evidence:\n" + str(item["evidence"])[:4000] +
            "\n\nQuestion:\n" + item["question"])

def prompt_naif(item):
    # Témoin négatif : ni rôle d'analyste, ni contexte 10-K, ni consigne de format.
    return item["question"]

print("prompt plein (extrait) :")
print(prompt_plein(items[0])[:220].replace("\n", " ⏎ "))
print()
print("prompt naïf (extrait)  :", prompt_naif(items[0])[:120])


prompt plein (extrait) :
You are a senior public-equities analyst. Answer the question using ONLY the evidence provided below. If the question asks for a number, reply with that number alone (same unit as the question asks). ⏎  ⏎ Evidence: ⏎ [{'eviden

prompt naïf (extrait)  : What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on th


### Lecture des prompts

Le **bras plein** donne au modèle ce que l'usage interne des carnets lui donne :
un rôle, le contexte documentaire, une consigne de format — puis le benchmark
jugé sur la réponse. Le **bras naïf** retire tout : la question nue. Si le bras
naïf scoraient aussi bien, l'étalon mesurerait la question, pas la compétence
financière du pipeline — c'est le contre-factuel qui donne sa signification à la
mesure.


In [5]:
# 5. Campagne — 3 bras x 50 items

def campagne(client, model, fabrique_prompt, libelle):
    print(f"== {libelle} ({model}) — {len(items)} items")
    rows = []
    for i, it in enumerate(items, 1):
        try:
            rep = appeler(client, model, fabrique_prompt(it))
            sc = score_reponse(it["answer"], rep)
        except Exception as e:
            rep, sc = None, None
            print(f"  [{i:2d}/50] ECHEC {type(e).__name__}: {str(e)[:80]}")
        rows.append({"id": it["financebench_id"], "company": it["company"],
                     "answer": it["answer"], "reponse": rep, "score": sc})
        if i % 10 == 0:
            print(f"  [{i:2d}/50] appels totaux : {APPELS['n']}")
    return rows

bras = {}
if OPENAI_OK:
    bras["luna-plein"] = campagne(client_oai, MODELE_LUNA, prompt_plein, "luna — bras plein")
    bras["luna-naif"]  = campagne(client_oai, MODELE_LUNA, prompt_naif,  "luna — témoin naïf")
if ANTHROPIC_OK:
    bras["sonnet-plein"] = campagne(client_anth, MODELE_SONNET, prompt_plein, "sonnet — bras plein")

with open(os.path.join(".tmp", "i26308_bench", "eval_results.jsonl"), "w", encoding="utf-8") as f:
    for nom, rows in bras.items():
        for r in rows:
            f.write(json.dumps({"bras": nom, **r}, ensure_ascii=False) + "\n")
print(f"\ncampagne terminée — {APPELS['n']} appels, {APPELS['echecs']} échecs — résultats hors dépôt (.tmp/)")


== luna — bras plein (gpt-5.6-luna) — 50 items


  [10/50] appels totaux : 10


  [20/50] appels totaux : 20


  [30/50] appels totaux : 30


  [40/50] appels totaux : 40


  [50/50] appels totaux : 50
== luna — témoin naïf (gpt-5.6-luna) — 50 items


... (6 outputs left out)

  [20/50] appels totaux : 120


  [30/50] appels totaux : 130


  [40/50] appels totaux : 140


  [50/50] appels totaux : 150

campagne terminée — 150 appels, 0 échecs — résultats hors dépôt (.tmp/)


### Lecture de la campagne

Trois bras, **150 appels** : `luna-plein` (l'organ de la série, contexte complet),
`luna-naif` (le même modèle, question nue — témoin négatif), `sonnet-plein`
(comparaison cross-LLM, seconde jambe de QC-Py-26). Les réponses brutes partent
dans `.tmp/` : le dépôt ne reçoit que le carnet exécuté et ses sorties, jamais le
dataset ni les bruts (licence).


In [6]:
# 6. Résultats

def recap(rows):
    scores = [r["score"] for r in rows if r["score"] is not None]
    echecs = sum(1 for r in rows if r["score"] is None and r["reponse"] is None)
    sans_nombre = sum(1 for r in rows if r["score"] is None and r["reponse"] is not None)
    return {"n": len(rows),
            "score": round(100 * sum(scores) / len(scores), 1) if scores else None,
            "echecs_api": echecs, "sans_nombre": sans_nombre}

for nom, rows in bras.items():
    s = recap(rows)
    print(f"{nom:13s} : {s['score']} %  (n={s['n']}, échecs API={s['echecs_api']}, "
          f"réponses sans nombre={s['sans_nombre']})")

lp, ln = recap(bras["luna-plein"]), recap(bras["luna-naif"])
print(f"\néccart plein − naïf (luna) : {lp['score'] - ln['score']} pts")
if "sonnet-plein" in bras:
    sp = recap(bras["sonnet-plein"])
    print(f"cross-LLM (luna vs sonnet, bras plein) : {lp['score']} % vs {sp['score']} %")


luna-plein    : 92.1 %  (n=50, échecs API=0, réponses sans nombre=12)
luna-naif     : 25.0 %  (n=50, échecs API=0, réponses sans nombre=46)
sonnet-plein  : 47.7 %  (n=50, échecs API=0, réponses sans nombre=6)

éccart plein − naïf (luna) : 67.1 pts
cross-LLM (luna vs sonnet, bras plein) : 92.1 % vs 47.7 %


### Lecture des résultats

La strate numérique intégrale (50 items, sans échantillonnage) donne, à
l'exécution du 2026-11-14 :

| Bras | Score | Réponses sans nombre |
|---|---|---|
| `luna-plein` (rôle + contexte 10-K + consigne) | **92,1 %** | 12 |
| `luna-naif` (question nue) | 25,0 % | 46 |
| `sonnet-plein` (même protocole) | 47,7 % | 6 |

**Le témoin négatif porte le sens de la mesure.** Le même modèle, question nue
(ni rôle, ni 10-K, ni format), s'effondre de 92,1 % à 25,0 % — 67,1 pts d'écart,
46 réponses sur 50 sans nombre. L'étalon mesure donc la compétence du pipeline
(rôle, contexte, format) et non la question seule : une question de 10-K posée à
nu ne se laisse pas répondre.

**Cross-LLM.** À protocole identique, `claude-sonnet-5-5` score 47,7 % contre
92,1 % pour `gpt-5.6-luna`. Pour l'ensemble multi-LLM de QC-Py-44, c'est un
repère **mesuré** de pondération des jambes : sur l'extraction numérique de 10-K,
luna est deux fois plus efficace que sonnet.

Les 12 réponses « sans nombre » du bras plein sont des refus ou des nuances où le
modèle ne s'engage pas sur un chiffre (tolérance déclarée : aucune).

**Verdict (compétence de domaine, pas stratégie trading).** `gpt-5.6-luna`,
l'organ des carnets de la série, voit sa compétence de raisonnement financier
numérique **fortement étayée** par un étalon indépendant — 92,1 % sur la strate
numérique intégrale de FinanceBench. L'étalon ne mesure pas la capacité
trading-signal propre des carnets (section « Limites »).

## Limites (déclarées, conformément au body de #26308)

- **Anglophone vs usage FR** : les items sont en anglais ; les carnets de la série
  produisent en français. L'étalon mesure la compétence de raisonnement financier
  des modèles sous-jacents, pas la couche de langue des carnets.
- **Strate numérique (50/150)** : les 100 items à réponse libre exigent un juge LLM
  (un organe distinct, avec son propre biais) — hors périmètre de ce carnet,
  candidate pour une suite.
- **Général vs spécifique** : FinanceBench mesure le raisonnement financier
  **général** (extraction de métriques sur 10-K) ; il ne mesure pas la capacité
  spécifique de **trading-signal** des carnets. Le verdict porté ci-dessus porte
  sur la compétence de **domaine**, pas sur la stratégie.
- **Période des 10-K** : documents 2018-2023 — antérieurs au coupure de
  pré-training des modèles récents ; un item « connu » du pré-training scorerait
  haut sans raisonnement. La strate entière est affectée au même biais, le
  classement relatif des bras reste significatif.
- **Coût** : 150 appels de complétion bornés (≤300 tokens) — de l'ordre de
  0,50 $ au tarif du 2026-11 (consigné dans l'issue).

## Suite

L'item de suite naturel (hors périmètre ici) : bras **réponse libre** avec juge
LLM pour les 100 items restants — à ouvrir en issue fille de #26308.

See #26308 · Part of #19306


[<< Sommaire QC](../README.md) | [Précédent : QC-Py-45-Noyau-Risque-Deterministe <<](./QC-Py-45-Noyau-Risque-Deterministe.ipynb)